# UMUD — Block 8 Train — LeViT

**GPU notebook** — single encoder **`levit_128s`** @ 200×5ep on gray55+line prep.

Outputs: `apo_gray55_line_200_levit128s.pkl`, `timing_report.csv`, `val_umud_report.csv` (val UMUD + mt_ok).


## Configuration


In [ ]:
# --- Block 8 single-encoder train ---
RANDOM_SEED = 42
ENCODER_SLUG = "levit128s"
ARCH = "levit_128s"
FAMILY = "LeViT"
EXPORT_NAME = "apo_gray55_line_200_levit128s.pkl"
DATASET_SLUG = "ucheozoemena/umud-aligned-apo-gray55-line-timing-200"
EPOCHS = 5

VALID_PCT = 0.20
STRATIFY_VAL_BY_RESOLUTION = True
BATCH_SIZE = 8
IMG_SIZE = 224
MM_PER_PIXEL = 0.075
USE_CLASS_WEIGHTS = True
APO_FG_WEIGHT = 15.0

print(f"Block 8 | {FAMILY} | arch={ARCH} | slug={ENCODER_SLUG} | epochs={EPOCHS} | dataset={DATASET_SLUG}")


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub
from fastai.vision.all import (
    AddMaskCodes,
    CrossEntropyLossFlat,
    Dice,
    IntToFloatTensor,
    PILImage,
    PILMask,
    RandomSplitter,
    Resize,
    TransformBlock,
    aug_transforms,
    get_image_files,
    resnet18,
    resnet34,
    resnet50,
    unet_learner,
)
from fastai.data.block import DataBlock
from fastai.data.transforms import IndexSplitter

DATASET_ROOT = Path(f"/kaggle/input/datasets/{DATASET_SLUG}")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_SLUG))
WORKING = Path("/kaggle/working")

print(f"Dataset root: {DATASET_ROOT} (exists={DATASET_ROOT.exists()})")


In [ ]:
def resolve_subdir(root: Path, name: str) -> Path:
    direct = root / name
    if direct.exists():
        return direct
    candidates = [p for p in root.rglob(name) if p.is_dir() and p.name == name]
    if not candidates:
        raise FileNotFoundError(f"Could not find {name}/ under {root}")
    return candidates[0]

IMG_DIR = resolve_subdir(DATASET_ROOT, "images")
MSK_DIR = resolve_subdir(DATASET_ROOT, "masks")
print(f"images: {IMG_DIR}")
print(f"masks: {MSK_DIR}")


In [ ]:
SEG_CODES = ["background", "structure"]

FASTAI_ENCODERS = {
    "resnet18": resnet18,
    "resnet34": resnet34,
    "resnet50": resnet50,
}


def open_image_pil(fn):
    gray = np.array(PILImage.create(fn))
    if gray.ndim == 3:
        gray = gray[..., 0]
    rgb = np.stack([gray, gray, gray], axis=-1).astype(np.uint8)
    return PILImage.create(rgb)


def open_mask_pil(fn):
    arr = np.array(PILImage.create(fn))
    if arr.ndim == 3:
        arr = arr[..., 0]
    binary = (arr > 0).astype(np.uint8)
    return PILMask.create(binary)


def stratified_train_valid_stems(
    stems: list[str],
    labels: list[str],
    valid_pct: float,
    seed: int,
) -> tuple[list[str], list[str]]:
    from collections import Counter
    import random

    counts = Counter(labels)
    # sklearn needs ≥2 per class; bucket rare native resolutions together.
    collapsed = [label if counts[label] >= 2 else "other" for label in labels]
    counts = Counter(collapsed)
    if min(counts.values()) < 2:
        collapsed = [
            label if counts[label] >= 2 else "other_bucket" for label in collapsed
        ]
        counts = Counter(collapsed)
    if min(counts.values()) >= 2:
        from sklearn.model_selection import train_test_split

        return train_test_split(
            stems,
            test_size=valid_pct,
            random_state=seed,
            stratify=collapsed,
        )

    rng = random.Random(seed)
    by_cohort: dict[str, list[str]] = {}
    for stem, label in zip(stems, labels):
        by_cohort.setdefault(label, []).append(stem)
    train: list[str] = []
    valid: list[str] = []
    for cohort_stems in by_cohort.values():
        rng.shuffle(cohort_stems)
        if len(cohort_stems) == 1:
            train.extend(cohort_stems)
            continue
        n_val = max(1, round(len(cohort_stems) * valid_pct))
        valid.extend(cohort_stems[:n_val])
        train.extend(cohort_stems[n_val:])
    print("Stratified sklearn failed — used per-cohort manual split")
    return train, valid


def make_dls(fnames, valid_pct=0.20, bs=8, seed=42, stratify_cohort: dict[str, str] | None = None):
    if stratify_cohort:
        stems = [Path(f).stem for f in fnames]
        labels = [stratify_cohort.get(s, "unknown") for s in stems]
        train_stems, valid_stems = stratified_train_valid_stems(
            stems, labels, valid_pct=valid_pct, seed=seed
        )
        valid_set = set(valid_stems)
        valid_idx = [i for i, s in enumerate(stems) if s in valid_set]
        splitter = IndexSplitter(valid_idx)
        print(
            f"Stratified val: {len(valid_stems)} images across "
            f"{len(set(labels))} resolution cohorts"
        )
    else:
        splitter = RandomSplitter(valid_pct=valid_pct, seed=seed)

    block = DataBlock(
        blocks=(
            TransformBlock(type_tfms=open_image_pil, batch_tfms=IntToFloatTensor),
            TransformBlock(
                type_tfms=open_mask_pil,
                item_tfms=AddMaskCodes(codes=SEG_CODES),
                batch_tfms=IntToFloatTensor,
            ),
        ),
        get_items=lambda _: fnames,
        get_x=lambda f: IMG_DIR / f.name,
        get_y=lambda f: MSK_DIR / f.name,
        splitter=splitter,
        item_tfms=Resize(IMG_SIZE),
        batch_tfms=aug_transforms(size=IMG_SIZE, min_scale=0.75, flip_vert=False, do_flip=True),
    )
    return block.dataloaders(fnames, bs=bs, num_workers=2)


def load_cohort_by_stem(root: Path) -> dict[str, str]:
    manifest_dir = resolve_subdir(root, "manifests")
    manifest_path = manifest_dir / "train_apo_gray55_line.csv"
    if not manifest_path.exists():
        print(f"No manifest at {manifest_path} — falling back to random val split")
        return {}
    manifest = pd.read_csv(manifest_path)
    if "resolution_cohort" in manifest.columns:
        col = "resolution_cohort"
    elif {"img_h", "img_w"}.issubset(manifest.columns):
        manifest["resolution_cohort"] = manifest.apply(
            lambda r: f"{int(r.img_h)}x{int(r.img_w)}", axis=1
        )
        col = "resolution_cohort"
    else:
        print("Manifest missing resolution columns — random val split")
        return {}
    return dict(zip(manifest["stem"].astype(str), manifest[col].astype(str)))


img_fnames = get_image_files(IMG_DIR)
msk_lookup = {p.name for p in get_image_files(MSK_DIR)}
fnames = [f for f in img_fnames if f.name in msk_lookup]
print(f"Pairs: {len(fnames)}")
assert len(fnames) > 0, "No image/mask pairs in mounted dataset"
cohort_by_stem = load_cohort_by_stem(DATASET_ROOT) if STRATIFY_VAL_BY_RESOLUTION else {}


In [ ]:
t0 = time.perf_counter()
dls = make_dls(
    fnames,
    valid_pct=VALID_PCT,
    bs=BATCH_SIZE,
    seed=RANDOM_SEED,
    stratify_cohort=cohort_by_stem or None,
)
_ = dls.one_batch()
print(f"Dataloader ready: {time.perf_counter() - t0:.1f}s")
dls.show_batch(max_n=4)


## Timm U-Net


In [ ]:
"""Timm-backed U-Net learner for fastai (adapted from walkwithfastai/wwf.vision.timm)."""
from __future__ import annotations

from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from timm import create_model
from fastai.torch_core import defaults

from fastai.layers import (
    BatchNorm,
    ConvLayer,
    MergeLayer,
    ResBlock,
    SigmoidRange,
    SimpleSelfAttention,
    apply_init,
)
from fastai.vision.models.unet import ResizeToOrig
from fastai.layers import PixelShuffle_ICNR
from fastai.vision.all import (
    Adam,
    Learner,
    Module,
    SequentialEx,
    get_c,
    ifnone,
    params,
)
from fastai.vision.learner import _add_norm
from fastai.basics import store_attr


class DecoderBlock(Module):
    def __init__(
        self,
        up_in_c,
        s_in_c,
        scale=2,
        blur=False,
        final_div=True,
        act_cls=defaults.activation,
        init=nn.init.kaiming_normal_,
        norm_type=None,
        **kwargs,
    ):
        self.shuf = PixelShuffle_ICNR(
            up_in_c,
            up_in_c // 2,
            scale=scale,
            blur=blur,
            act_cls=act_cls,
            norm_type=norm_type,
            **kwargs,
        )
        self.bn = BatchNorm(s_in_c)
        self.act = act_cls()
        ni = up_in_c // 2 + s_in_c
        nf = ni if final_div else ni // 2
        self.nf = nf
        self.conv1 = ConvLayer(ni, nf, act_cls=act_cls, norm_type=norm_type, **kwargs)
        self.conv2 = ConvLayer(nf, nf, act_cls=act_cls, norm_type=norm_type, **kwargs)
        apply_init(nn.Sequential(self.shuf, self.bn, self.conv1, self.conv2), init)

    def forward(self, up_in: torch.Tensor, skip: Optional[torch.Tensor] = None):
        x = self.shuf(up_in)
        if skip is not None:
            ssh = skip.shape[-2:]
            if ssh != x.shape[-2:]:
                x = F.interpolate(x, ssh, mode="nearest")
            x = self.act(torch.cat([x, self.bn(skip)], dim=1))
        return self.conv2(self.conv1(x))


def BatchNormZero(nf, ndim=2, **kwargs):
    from fastai.layers import _get_norm

    return _get_norm("BatchNorm", nf, ndim, zero=True, **kwargs)


def _make_bottleneck(
    bottleneck,
    ni,
    act_cls=defaults.activation,
    norm_type=None,
    init=nn.init.kaiming_normal_,
    **kwargs,
):
    if bottleneck == "conv":
        seq = nn.Sequential(
            BatchNorm(ni),
            nn.ReLU(),
            ConvLayer(ni, ni * 2, act_cls=act_cls, norm_type=norm_type, **kwargs),
            ConvLayer(ni * 2, ni, act_cls=act_cls, norm_type=norm_type, **kwargs),
        )
    elif bottleneck == "attention":
        seq = nn.Sequential(
            BatchNormZero(ni),
            nn.ReLU(),
            ConvLayer(ni, ni, act_cls=act_cls, norm_type=norm_type, **kwargs),
            SimpleSelfAttention(ni, ks=1),
        )
    else:
        raise NotImplementedError(f"Bottleneck architecture {bottleneck} not implemented.")
    apply_init(seq, init)
    return seq


class UnetDecoder(Module):
    def __init__(
        self,
        encoder,
        bottleneck=None,
        blur=False,
        blur_final=True,
        norm_type=None,
        act_cls=defaults.activation,
        init=nn.init.kaiming_normal_,
        **kwargs,
    ):
        encoder_chs = encoder.feature_info.channels()[::-1]
        encoder_reds = encoder.feature_info.reduction()
        skip_channels = encoder_chs[1:]
        self.blocks = nn.ModuleList()
        up_c = encoder_chs[0]
        self.bottleneck = (
            _make_bottleneck(bottleneck, up_c, act_cls=act_cls, norm_type=norm_type, init=init, **kwargs)
            if isinstance(bottleneck, str)
            else bottleneck
        )
        for i, skip_c in enumerate(skip_channels):
            not_final = i != len(skip_channels) - 1
            do_blur = blur and (not_final or blur_final)
            scale = encoder_reds.pop() // encoder_reds[-1]
            block = DecoderBlock(
                up_c,
                skip_c,
                scale=scale,
                blur=do_blur,
                final_div=not_final,
                act_cls=act_cls,
                init=init,
                norm_type=norm_type,
                **kwargs,
            )
            up_c = block.nf
            self.blocks.append(block)
        scale = encoder_reds[0]
        self.final_shuf = (
            PixelShuffle_ICNR(up_c, scale=scale, act_cls=act_cls, norm_type=norm_type, **kwargs)
            if scale != 1
            else None
        )
        self.nf = up_c

    def forward(self, x: List[torch.Tensor]):
        x.reverse()
        skips = x[1:]
        x = x[0]
        if self.bottleneck is not None:
            x = self.bottleneck(x)
        for i, b in enumerate(self.blocks):
            skip = skips[i] if i < len(skips) else None
            x = b(x, skip)
        if self.final_shuf is not None:
            x = self.final_shuf(x)
        return x


class UnetHead(SequentialEx):
    def __init__(
        self,
        up_c,
        n_out,
        last_cross=False,
        n_in=None,
        bottle=False,
        norm_type=None,
        act_cls=defaults.activation,
        init=nn.init.kaiming_normal_,
        y_range=None,
        **kwargs,
    ):
        layers = nn.ModuleList([ResizeToOrig()])
        if last_cross:
            if n_in is None:
                raise AttributeError("You must specify `n_in` if `last_cross=True`.")
            up_c += n_in
            layers.extend(
                [
                    MergeLayer(dense=True),
                    ResBlock(
                        1,
                        up_c,
                        up_c // 2 if bottle else up_c,
                        act_cls=act_cls,
                        norm_type=norm_type,
                        **kwargs,
                    ),
                ]
            )
        layers.append(nn.Conv2d(up_c, n_out, kernel_size=(1, 1)))
        if y_range is not None:
            layers.append(SigmoidRange(*y_range))
        super().__init__(*layers)
        apply_init(self, init)


class TimmUnet(SequentialEx):
    def __init__(
        self,
        encoder="resnet50",
        n_in=3,
        n_out=1,
        encoder_kwargs=None,
        encoder_indices=None,
        blur=False,
        blur_final=True,
        bottleneck=None,
        last_cross=True,
        norm_type=None,
        act_cls=defaults.activation,
        init=nn.init.kaiming_normal_,
        pretrained=True,
        y_range=None,
        bottle=False,
        **kwargs,
    ):
        encoder_kwargs = encoder_kwargs or {}
        self.encoder = create_model(
            encoder,
            features_only=True,
            out_indices=encoder_indices,
            in_chans=n_in,
            pretrained=pretrained,
            **encoder_kwargs,
        )
        self.decoder = UnetDecoder(
            self.encoder,
            bottleneck=bottleneck,
            blur=blur,
            blur_final=blur_final,
            norm_type=norm_type,
            act_cls=act_cls,
            init=init,
            **kwargs,
        )
        self.head = UnetHead(
            self.decoder.nf,
            n_out,
            last_cross=last_cross,
            n_in=n_in,
            bottle=bottle,
            norm_type=norm_type,
            act_cls=act_cls,
            init=init,
            y_range=y_range,
            **kwargs,
        )
        super().__init__(nn.Sequential(self.encoder, self.decoder), *self.head)

    @property
    def default_cfg(self):
        return self.encoder.default_cfg

    @property
    def feature_info(self):
        return self.encoder.feature_info


def _get_params_from_attrs(m, ls):
    from fastai.torch_core import getattrs

    return params(nn.Sequential(*getattrs(m, *ls)))


def _get_params_from_modules(modules):
    return params(nn.Sequential(*modules))


def split_nested_list(l, idxs, left_in=None, right_in=None):
    from fastai.data.core import L

    left_in, right_in = ifnone(left_in, L()), ifnone(right_in, L())
    idx = int(idxs[0])
    if len(idxs) == 1:
        left, right = l[:idx], l[idx:]
    else:
        left, right = split_nested_list(l[idx], idxs[1:], l[:idx], l[idx + 1 :])
    return L(L(*left_in, *left), L(*right, *right_in))


def _timm_splitter(m):
    from fastai.torch_core import getattrs
    from fastai.data.core import L

    split_path = m.feature_info.module_name(0).split(".")

    def module_children(mod):
        return getattrs(mod, *list(mod._modules.keys()))

    enc = m.encoder
    enc_names = list(enc._modules.keys())
    enc_mods = module_children(enc)

    # LeViT-style single wrapper
    if len(enc_mods) == 1 and enc_mods[0]._modules:
        inner = enc_mods[0]
        inner_names = list(inner._modules.keys())
        inner_mods = module_children(inner)
        cut = _split_cut_index(inner_names, split_path)
        encoder_early = _get_params_from_modules(inner_mods[:cut])
        encoder_late = _get_params_from_modules(inner_mods[cut:])
    else:
        cut = _split_cut_index(enc_names, split_path)
        encoder_early = _get_params_from_modules(enc_mods[:cut])
        encoder_late = _get_params_from_modules(enc_mods[cut:])

    decoder = _get_params_from_attrs(m.decoder, L(*list(m.decoder._modules.keys())))
    head = _get_params_from_attrs(m.head, L(*list(m.head._modules.keys())))
    return L(encoder_early, encoder_late, L(*decoder, *head))


def _split_cut_index(names: list[str], split_path: list[str]) -> int:
    """Index in `names` where fine-tuning should begin (modules before are frozen)."""
    if not split_path:
        return max(1, len(names) // 2)
    head = split_path[0]
    if head in names:
        return names.index(head)
    if len(split_path) >= 2 and split_path[1].isdigit():
        flat = f"{head}_{split_path[1]}"
        if flat in names:
            return names.index(flat)
    flat = "_".join(split_path[:2]) if len(split_path) >= 2 else head
    if flat in names:
        return names.index(flat)
    for i, n in enumerate(names):
        if n.startswith(head):
            return i
    return max(1, len(names) // 2)


def _timm_stats(m):
    return tuple((list(m.default_cfg[stat]) for stat in ("mean", "std")))


def timm_unet_learner(
    dls,
    arch,
    normalize=True,
    n_in=None,
    n_out=None,
    pretrained=True,
    loss_func=None,
    opt_func=Adam,
    lr=defaults.lr,
    splitter=None,
    cbs=None,
    metrics=None,
    path=None,
    model_dir="models",
    wd=None,
    wd_bn_bias=False,
    train_bn=True,
    moms=(0.95, 0.85, 0.95),
    bottleneck="conv",
    **kwargs,
):
    """Build a U-Net learner with a timm encoder."""
    n_in = ifnone(n_in, dls.one_batch()[0].shape[1])
    n_out = ifnone(n_out, get_c(dls))
    assert n_out, "`n_out` is not defined — set `dls.c` or pass `n_out`"
    model = TimmUnet(arch, n_in=n_in, n_out=n_out, pretrained=pretrained, bottleneck=bottleneck, **kwargs)
    if normalize:
        _add_norm(dls, {"stats": _timm_stats(model)}, pretrained)
    splitter = ifnone(splitter, _timm_splitter)
    learn = Learner(
        dls=dls,
        model=model,
        loss_func=loss_func,
        opt_func=opt_func,
        lr=lr,
        splitter=splitter,
        cbs=cbs,
        metrics=metrics,
        path=path,
        model_dir=model_dir,
        wd=wd,
        wd_bn_bias=wd_bn_bias,
        train_bn=train_bn,
        moms=moms,
    )
    if pretrained:
        learn.freeze()
    store_attr("arch,normalize,n_out,pretrained", self=learn, **kwargs)
    return learn


In [ ]:
t_train = time.perf_counter()
import torch
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)

if USE_CLASS_WEIGHTS:
    loss_weights = torch.tensor([1.0, APO_FG_WEIGHT])
    loss_func = CrossEntropyLossFlat(axis=1, weight=loss_weights)
    print(f"Class weights: background=1.0, structure={APO_FG_WEIGHT}")
else:
    loss_func = CrossEntropyLossFlat(axis=1)

learn = timm_unet_learner(
    dls,
    ARCH,
    metrics=[Dice()],
    loss_func=loss_func,
    bottleneck="conv",
)
learn.fine_tune(EPOCHS)
train_sec = time.perf_counter() - t_train
print(f"Train wall-clock: {train_sec:.1f}s")
learn.export(WORKING / EXPORT_NAME)

val_losses, val_metrics = learn.validate(dl=dls.valid)
if isinstance(val_metrics, (list, tuple)):
    val_dice = float(val_metrics[0]) if val_metrics else float("nan")
else:
    val_dice = float(val_metrics)
print(f"Val Dice (reference): {val_dice:.4f}")

timing = pd.DataFrame(
    [
        {
            "encoder_slug": ENCODER_SLUG,
            "family": FAMILY,
            "arch": ARCH,
            "n_pairs": len(fnames),
            "epochs": EPOCHS,
            "img_size": IMG_SIZE,
            "val_dice": round(val_dice, 4) if val_dice == val_dice else None,
            "total_sec": round(train_sec, 1),
            "sec_per_pair_epoch": round(train_sec / max(1, len(fnames) * EPOCHS), 3),
            "dataset": DATASET_SLUG,
        }
    ]
)
timing.to_csv(WORKING / "timing_report.csv", index=False)
display(timing)


## Val UMUD score (primary model-selection metric)

End-to-end **segment-then-measure** on the stratified val split:

- **GT:** stretch-aligned fasc + line-converted apo masks → PA/FL/MT @ `MM_PER_PIXEL`
- **Pred:** production **fasc** model + **trained apo** (gray55 infer, horiz_parallel)
- **Metric:** official UMUD score (`scripts/umud_score.py`) — **lower is better**

Also reports `val_mt_ok_pct` (% val images with finite PA/FL/MT) — must reach **100%** before test submit.


In [ ]:
"""Segment-then-measure geometry for UMUD (shared by submission + train eval)."""
from __future__ import annotations

import cv2
import numpy as np
from PIL import Image

APO_REGION_THRESHOLD = 0.50
GRAY_FILL_VALUE = 55
ROI_THRESH = 5
ROI_PAD_PX = 10
TOP_K_CANDIDATES = 8
MIN_SEP_PX = 15
REGION_STYLE_THRESHOLD = 0.50
LINE_THICKNESS = 3


def load_gray(path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr.mean(axis=-1)
    return arr.astype(np.uint8)


def load_mask(path) -> np.ndarray:
    with Image.open(path) as im:
        arr = np.array(im)
    if arr.ndim == 3:
        arr = arr[..., 0]
    return (arr > 0).astype(np.uint8)


def align_mask(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return mask
    return (
        np.array(
            Image.fromarray((mask * 255).astype(np.uint8)).resize(
                (target_w, target_h), Image.NEAREST
            )
        )
        > 0
    ).astype(np.uint8)


def region_to_line_mask(region: np.ndarray, thickness: int = LINE_THICKNESS) -> np.ndarray:
    h, w = region.shape
    out = np.zeros((h, w), dtype=np.uint8)
    top_pts, bot_pts = [], []
    for x in range(w):
        idx = np.where(region[:, x] > 0)[0]
        if len(idx) == 0:
            continue
        top_pts.append((x, int(idx.min())))
        bot_pts.append((x, int(idx.max())))
    if not top_pts:
        return out
    cv2.polylines(out, [np.array(top_pts, dtype=np.int32).reshape(-1, 1, 2)], False, 1, thickness)
    cv2.polylines(out, [np.array(bot_pts, dtype=np.int32).reshape(-1, 1, 2)], False, 1, thickness)
    return out


def to_line_target(mask: np.ndarray) -> tuple[np.ndarray, str, bool]:
    cov = float(mask.mean())
    if cov >= REGION_STYLE_THRESHOLD:
        return region_to_line_mask(mask), "region", True
    return mask, "line", False


def resize_image(img: np.ndarray, size: int) -> np.ndarray:
    return np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def resize_mask_to(mask: np.ndarray, target_h: int, target_w: int) -> np.ndarray:
    if mask.shape == (target_h, target_w):
        return (mask > 0).astype(np.uint8)
    src = (mask > 0).astype(np.uint8) * 255
    out = Image.fromarray(src).resize((target_w, target_h), Image.NEAREST)
    return (np.array(out) > 0).astype(np.uint8)


def tensor_to_mask(pred) -> np.ndarray:
    if hasattr(pred, "cpu"):
        pred = pred.cpu().numpy()
    arr = np.asarray(pred)
    if arr.ndim == 3:
        arr = arr.argmax(axis=0)
    return (arr > 0).astype(np.uint8)


def tag_apo_style(coverage: float) -> str:
    return "region" if coverage >= APO_REGION_THRESHOLD else "line"


def invert_mask(mask: np.ndarray) -> np.ndarray:
    return (1 - mask).astype(np.uint8)


def effective_apo_mask(mask: np.ndarray, style: str) -> tuple[np.ndarray, str]:
    if style == "region":
        return invert_mask(mask), "inverted_region"
    return mask, "raw_line"


def find_apo_contours(mask: np.ndarray, min_area_frac: float = 0.0003) -> list[np.ndarray]:
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = mask.size * min_area_frac
    big = [c for c in contours if cv2.contourArea(c) >= min_area]
    big.sort(key=lambda c: cv2.boundingRect(c)[1])
    return big


def find_roi_bbox(img_gray: np.ndarray, thr: float = ROI_THRESH, pad: int = ROI_PAD_PX):
    roi = (img_gray > thr).astype(np.uint8)
    if roi.sum() == 0:
        h, w = img_gray.shape
        return 0, h, 0, w
    num, labels, stats, _ = cv2.connectedComponentsWithStats(roi, connectivity=8)
    best = max(range(1, num), key=lambda i: stats[i, cv2.CC_STAT_AREA])
    x, y, w, h = stats[best, :4]
    y0 = max(0, y - pad)
    y1 = min(img_gray.shape[0], y + h + pad)
    x0 = max(0, x - pad)
    x1 = min(img_gray.shape[1], x + w + pad)
    return int(y0), int(y1), int(x0), int(x1)


def preprocess_gray55(img_native: np.ndarray):
    bbox = find_roi_bbox(img_native)
    y0, y1, x0, x1 = bbox
    pre = img_native.copy()
    outside = np.ones(img_native.shape, dtype=bool)
    outside[y0:y1, x0:x1] = False
    pre[outside] = GRAY_FILL_VALUE
    return pre, bbox


def clip_mask_to_bbox(mask: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    y0, y1, x0, x1 = bbox
    out = np.zeros_like(mask, dtype=np.uint8)
    out[y0:y1, x0:x1] = mask[y0:y1, x0:x1]
    return out


def edge_polyline(contour: np.ndarray, which: str = "bottom", n_bins: int = 60):
    pts = contour.reshape(-1, 2)
    if len(pts) < 3:
        return np.array([]), np.array([])
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    if x_max <= x_min:
        return pts[:, 0].astype(float), pts[:, 1].astype(float)
    edges = np.linspace(x_min, x_max, n_bins + 1)
    xs_out, ys_out = [], []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = pts[(pts[:, 0] >= lo) & (pts[:, 0] < hi)]
        if len(in_bin) == 0:
            continue
        y = in_bin[:, 1].max() if which == "bottom" else in_bin[:, 1].min()
        xs_out.append((lo + hi) / 2.0)
        ys_out.append(float(y))
    return np.array(xs_out), np.array(ys_out)


def fit_line(xs: np.ndarray, ys: np.ndarray):
    if len(xs) < 2:
        return None
    return np.poly1d(np.polyfit(xs, ys, 1))


def line_angle_deg(line) -> float:
    return float(np.degrees(np.arctan(line[1]))) if line is not None else np.nan


def mt_from_apo_edges(sup_line, deep_line, x_left: float, x_right: float) -> float:
    if sup_line is None or deep_line is None or x_right <= x_left:
        return np.nan
    thirds = [x_left + (x_right - x_left) * t for t in (1 / 6, 3 / 6, 5 / 6)]
    dists = [abs(float(deep_line(x) - sup_line(x))) for x in thirds]
    return float(np.mean(dists))


def fascicle_pca(mask: np.ndarray) -> dict | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < 3:
        return None
    coords = np.column_stack([xs.astype(float), ys.astype(float)])
    centered = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    direction = vh[0]
    projections = centered @ direction
    return {
        "length_px": float(projections.max() - projections.min()),
        "angle_deg": float(np.degrees(np.arctan2(direction[1], direction[0]))),
    }


def acute_angle_deg(a1: float, a2: float) -> float:
    d = abs(a1 - a2) % 180.0
    return float(min(d, 180.0 - d))


def contour_feats(c: np.ndarray) -> dict:
    x, y, w, h = cv2.boundingRect(c)
    pts = c.reshape(-1, 2)
    x_span = float(pts[:, 0].max() - pts[:, 0].min())
    area = float(cv2.contourArea(c))
    return {"area": area, "x_span": x_span, "y_top": y, "y_bot": y + h, "w": w, "h": h}


def x_overlap_from_contours(sup_c: np.ndarray, deep_c: np.ndarray) -> float:
    sup_x, _ = edge_polyline(sup_c, which="bottom")
    deep_x, _ = edge_polyline(deep_c, which="top")
    if len(sup_x) == 0 or len(deep_x) == 0:
        return 0.0
    return max(0.0, min(float(sup_x.max()), float(deep_x.max())) - max(float(sup_x.min()), float(deep_x.min())))


def edge_angle_from_horizontal(contour: np.ndarray, which: str):
    xs, ys = edge_polyline(contour, which=which)
    if len(xs) < 2:
        return None
    line = fit_line(xs, ys)
    if line is None:
        return None
    ang = abs(float(np.degrees(np.arctan(line[1])))) % 180.0
    return float(min(ang, 180.0 - ang))


def horizontality_factor(angle_deg) -> float:
    if angle_deg is None:
        return 0.0
    return float(np.cos(np.radians(angle_deg)) ** 2)


def parallelism_factor(sup_ang, deep_ang) -> float:
    if sup_ang is None or deep_ang is None:
        return 0.0
    d = abs(sup_ang - deep_ang) % 180.0
    d = min(d, 180.0 - d)
    return float(np.cos(np.radians(d)) ** 2)


def pick_best_pair_xspan(contours: list[np.ndarray], min_sep_px: int = MIN_SEP_PX, top_k: int = TOP_K_CANDIDATES):
    if len(contours) < 2:
        return None, None
    ranked = sorted(contours, key=lambda c: contour_feats(c)["x_span"], reverse=True)
    candidates = ranked[: min(top_k, len(ranked))]
    best_pair = None
    best_overlap = -1.0
    for i, ci in enumerate(candidates):
        for cj in candidates[i + 1 :]:
            fi, fj = contour_feats(ci), contour_feats(cj)
            if fi["y_top"] <= fj["y_top"]:
                sup_c, deep_c = ci, cj
                fs, fd = fi, fj
            else:
                sup_c, deep_c = cj, ci
                fs, fd = fj, fi
            if fd["y_top"] < fs["y_top"] + min_sep_px:
                continue
            overlap = x_overlap_from_contours(sup_c, deep_c)
            if overlap > best_overlap:
                best_overlap = overlap
                best_pair = (sup_c, deep_c)
    if best_pair is not None:
        return best_pair
    if len(candidates) >= 2:
        top2 = candidates[:2]
        top2.sort(key=lambda c: contour_feats(c)["y_top"])
        return top2[0], top2[1]
    return None, None


def pick_horiz_parallel(contours: list[np.ndarray], min_sep_px: int = MIN_SEP_PX, top_k: int = TOP_K_CANDIDATES):
    if len(contours) < 2:
        return None, None
    ranked = sorted(
        contours,
        key=lambda c: max(
            contour_feats(c)["x_span"] * horizontality_factor(edge_angle_from_horizontal(c, "bottom")),
            contour_feats(c)["x_span"] * horizontality_factor(edge_angle_from_horizontal(c, "top")),
        ),
        reverse=True,
    )
    candidates = ranked[: min(top_k, len(ranked))]
    best_pair = None
    best_score = -1.0
    for i, ci in enumerate(candidates):
        for cj in candidates[i + 1 :]:
            fi, fj = contour_feats(ci), contour_feats(cj)
            if fi["y_top"] <= fj["y_top"]:
                sup_c, deep_c = ci, cj
                fs, fd = fi, fj
            else:
                sup_c, deep_c = cj, ci
                fs, fd = fj, fi
            if fd["y_top"] < fs["y_top"] + min_sep_px:
                continue
            overlap = x_overlap_from_contours(sup_c, deep_c)
            if overlap <= 0:
                continue
            sup_ang = edge_angle_from_horizontal(sup_c, "bottom")
            deep_ang = edge_angle_from_horizontal(deep_c, "top")
            score = (
                overlap
                * horizontality_factor(sup_ang)
                * horizontality_factor(deep_ang)
                * parallelism_factor(sup_ang, deep_ang)
            )
            if score > best_score:
                best_score = score
                best_pair = (sup_c, deep_c)
    if best_pair is not None:
        return best_pair
    return pick_best_pair_xspan(contours, min_sep_px=min_sep_px, top_k=top_k)


def pick_superficial_deep(contours: list[np.ndarray], min_sep_px: int = 15):
    sup_c, deep_c = pick_horiz_parallel(contours, min_sep_px=min_sep_px, top_k=TOP_K_CANDIDATES)
    return sup_c, deep_c, len(contours)


def apo_geometry_attempt(apo_mask: np.ndarray, style: str) -> dict:
    eff, method = effective_apo_mask(apo_mask, style)
    contours = find_apo_contours(eff)
    sup_c, deep_c, n_contours = pick_superficial_deep(contours)
    out = {
        "apo_method": method,
        "n_contours": n_contours,
        "mt_px": np.nan,
        "deep_angle_deg": np.nan,
        "mt_fail_reason": "ok",
    }
    if len(contours) == 0:
        out["mt_fail_reason"] = "no_contours"
        return out
    if n_contours < 2 or sup_c is None or deep_c is None:
        out["mt_fail_reason"] = "single_contour"
        return out
    sup_x, sup_y = edge_polyline(sup_c, which="bottom")
    deep_x, deep_y = edge_polyline(deep_c, which="top")
    sup_line = fit_line(sup_x, sup_y)
    deep_line = fit_line(deep_x, deep_y)
    if sup_line is None or deep_line is None:
        out["mt_fail_reason"] = "line_fit_fail"
        return out
    if len(sup_x) == 0 or len(deep_x) == 0:
        out["mt_fail_reason"] = "empty_edge_polyline"
        return out
    x_left = max(sup_x.min(), deep_x.min())
    x_right = min(sup_x.max(), deep_x.max())
    if x_right <= x_left:
        out["mt_fail_reason"] = "no_x_overlap"
        return out
    out["mt_px"] = mt_from_apo_edges(sup_line, deep_line, x_left, x_right)
    out["deep_angle_deg"] = line_angle_deg(deep_line)
    if np.isnan(out["mt_px"]):
        out["mt_fail_reason"] = "mt_compute_nan"
    return out


def apo_geometry_from_mask(apo_mask: np.ndarray, style: str) -> dict:
    primary = apo_geometry_attempt(apo_mask, style)
    if not np.isnan(primary["mt_px"]):
        return primary
    if style == "region":
        fallback = apo_geometry_attempt(apo_mask, "line")
        if not np.isnan(fallback["mt_px"]):
            fallback["apo_method"] = f"{primary['apo_method']}+fallback_line"
            fallback["mt_fail_reason_primary"] = primary["mt_fail_reason"]
            return fallback
    primary["mt_fail_reason_primary"] = primary["mt_fail_reason"]
    return primary


def derive_geometry(fasc_mask: np.ndarray, apo_mask: np.ndarray, apo_style: str) -> dict:
    apo = apo_geometry_from_mask(apo_mask, apo_style)
    fpca = fascicle_pca(fasc_mask)
    out = {
        "pa_deg": np.nan,
        "fl_px": np.nan,
        "mt_px": apo["mt_px"],
        "mt_fail_reason": apo.get("mt_fail_reason"),
        "apo_cov": float(apo_mask.mean()),
    }
    if fpca is not None:
        out["fl_px"] = fpca["length_px"]
        ref = apo["deep_angle_deg"] if not np.isnan(apo["deep_angle_deg"]) else 0.0
        out["pa_deg"] = acute_angle_deg(fpca["angle_deg"], ref)
    return out


def geometry_to_mm(geo: dict, mm_per_pixel: float) -> dict:
    pa = geo["pa_deg"]
    fl_mm = geo["fl_px"] * mm_per_pixel if not np.isnan(geo["fl_px"]) else np.nan
    mt_mm = geo["mt_px"] * mm_per_pixel if not np.isnan(geo["mt_px"]) else np.nan
    return {"pa_deg": pa, "fl_mm": fl_mm, "mt_mm": mt_mm}


def gt_geometry_from_masks(
    fasc_mask_raw: np.ndarray,
    apo_mask_raw: np.ndarray,
    img_shape: tuple[int, int],
    mm_per_pixel: float,
) -> dict:
    h, w = img_shape
    fasc = align_mask(fasc_mask_raw, h, w)
    apo_aligned = align_mask(apo_mask_raw, h, w)
    apo_line, _, _ = to_line_target(apo_aligned)
    geo = derive_geometry(fasc, apo_line, "line")
    return geometry_to_mm(geo, mm_per_pixel)


def predict_geometry(
    img_native: np.ndarray,
    fasc_learn,
    apo_learn,
    img_size: int,
    mm_per_pixel: float,
):
    from fastai.vision.all import PILImage

    img_gray55, bbox = preprocess_gray55(img_native)
    h, w = img_native.shape

    def open_rgb_256(img: np.ndarray) -> PILImage:
        small = resize_image(img, img_size)
        rgb = np.stack([small, small, small], axis=-1).astype(np.uint8)
        return PILImage.create(rgb)

    _, fasc_t, _ = fasc_learn.predict(open_rgb_256(img_native))
    _, apo_t, _ = apo_learn.predict(open_rgb_256(img_gray55))
    fasc_native = resize_mask_to(tensor_to_mask(fasc_t), h, w)
    apo_native = clip_mask_to_bbox(resize_mask_to(tensor_to_mask(apo_t), h, w), bbox)
    apo_style = tag_apo_style(float(apo_native.mean()))
    geo = derive_geometry(fasc_native, apo_native, apo_style)
    mm = geometry_to_mm(geo, mm_per_pixel)
    mm["mt_fail_reason"] = geo.get("mt_fail_reason")
    mm["apo_cov"] = geo.get("apo_cov")
    return mm


"""Official UMUD leaderboard metric (from paulritsche/umud-score on Kaggle).

Use with a ground-truth DataFrame (train geometry or hidden test labels) and a
submission-shaped prediction DataFrame. Requires no NaNs in predictions.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

TARGET_COLS = ("pa_deg", "fl_mm", "mt_mm")

TOLERANCES = {
    "pa_deg": 6.0,
    "fl_mm": 12.0,
    "mt_mm": 3.0,
}

WEIGHTS = {
    "pa_deg": 1.0,
    "fl_mm": 1.0,
    "mt_mm": 1.0,
}

EPS_SECONDARY = 1e-6
EPS_TERTIARY = 1e-9


class MetricError(ValueError):
    """Malformed submission for UMUD scoring."""


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str = "image_id",
) -> float:
    """Return official composite score (lower is better)."""
    if row_id_column_name not in solution.columns:
        raise MetricError(f"Solution missing id column '{row_id_column_name}'.")
    if row_id_column_name not in submission.columns:
        raise MetricError(f"Submission missing id column '{row_id_column_name}'.")

    for col in TARGET_COLS:
        if col not in solution.columns:
            raise MetricError(f"Solution missing ground-truth column '{col}'.")
        if col not in submission.columns:
            raise MetricError(f"Submission missing prediction column '{col}'.")

    if solution[row_id_column_name].duplicated().any():
        raise MetricError(f"Duplicate ids in solution '{row_id_column_name}'.")
    if submission[row_id_column_name].duplicated().any():
        raise MetricError(f"Duplicate ids in submission '{row_id_column_name}'.")

    merged = solution.merge(
        submission,
        on=row_id_column_name,
        how="inner",
        suffixes=("_true", "_pred"),
    )
    if len(merged) != len(solution):
        missing = len(solution) - len(merged)
        raise MetricError(f"Submission missing {missing} required row ids.")

    for col in TARGET_COLS:
        pred_col = f"{col}_pred"
        merged[pred_col] = pd.to_numeric(merged[pred_col], errors="coerce")
        if merged[pred_col].isna().any():
            raise MetricError(f"Column '{col}' must be numeric with no NaN values.")
        if np.isinf(merged[pred_col].to_numpy()).any():
            raise MetricError(f"Column '{col}' contains infinite values.")

    weight_sum = float(sum(WEIGHTS[c] for c in TARGET_COLS))
    primary = secondary = tertiary = 0.0

    for col in TARGET_COLS:
        y_true = merged[f"{col}_true"].to_numpy(dtype=float)
        y_pred = merged[f"{col}_pred"].to_numpy(dtype=float)
        tau = float(TOLERANCES[col])
        w = float(WEIGHTS[col])
        if not np.isfinite(tau) or tau <= 0:
            raise MetricError(f"Tolerance for '{col}' must be positive.")

        primary += w * float(np.mean(np.abs(y_pred - y_true))) / tau
        secondary += w * float(np.median(np.abs(y_pred - y_true))) / tau
        tertiary += w * float(np.sqrt(np.mean((y_pred - y_true) ** 2))) / tau

    primary /= weight_sum
    secondary /= weight_sum
    tertiary /= weight_sum
    return float(primary + EPS_SECONDARY * secondary + EPS_TERTIARY * tertiary)


def local_metric_report(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str = "image_id",
) -> pd.DataFrame:
    """Per-target MAE / MedAE / RMSE / bias for local inspection."""
    merged = solution.merge(
        submission,
        on=row_id_column_name,
        how="inner",
        suffixes=("_true", "_pred"),
    )
    rows = []
    for col in TARGET_COLS:
        y_true = merged[f"{col}_true"].to_numpy(dtype=float)
        y_pred = merged[f"{col}_pred"].to_numpy(dtype=float)
        err = y_pred - y_true
        rows.append(
            {
                "target": col,
                "mae": float(np.mean(np.abs(err))),
                "medae": float(np.median(np.abs(err))),
                "rmse": float(np.sqrt(np.mean(err**2))),
                "bias": float(np.mean(err)),
                "nmae": float(np.mean(np.abs(err)) / TOLERANCES[col]),
            }
        )
    return pd.DataFrame(rows)


def score_summary(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str = "image_id",
) -> dict:
    """Score val predictions; report partial + strict leaderboard-style metrics."""
    merged = solution.merge(
        submission,
        on=row_id_column_name,
        how="inner",
        suffixes=("_true", "_pred"),
    )
    if len(merged) == 0:
        return {
            "n_total": 0,
            "n_pred_finite": 0,
            "n_gt_finite": 0,
            "n_scorable": 0,
            "val_mt_ok_pct": float("nan"),
            "val_umud_score": float("nan"),
            "val_umud_score_strict": float("nan"),
        }

    pred_finite = merged[[f"{c}_pred" for c in TARGET_COLS]].notna().all(axis=1)
    gt_finite = merged[[f"{c}_true" for c in TARGET_COLS]].notna().all(axis=1)
    scorable = pred_finite & gt_finite

    out = {
        "n_total": int(len(merged)),
        "n_pred_finite": int(pred_finite.sum()),
        "n_gt_finite": int(gt_finite.sum()),
        "n_scorable": int(scorable.sum()),
        "val_mt_ok_pct": round(100.0 * float(pred_finite.mean()), 2),
        "val_umud_score": float("nan"),
        "val_umud_score_strict": float("nan"),
    }

    if scorable.any():
        sol = merged.loc[scorable, [row_id_column_name]].copy()
        sub = merged.loc[scorable, [row_id_column_name]].copy()
        for col in TARGET_COLS:
            sol[col] = merged.loc[scorable, f"{col}_true"].astype(float)
            sub[col] = merged.loc[scorable, f"{col}_pred"].astype(float)
        out["val_umud_score"] = round(float(score(sol, sub, row_id_column_name)), 6)

    if pred_finite.all():
        sub_all = submission[[row_id_column_name, *TARGET_COLS]].copy()
        sol_all = solution[[row_id_column_name, *TARGET_COLS]].copy()
        out["val_umud_score_strict"] = round(
            float(score(sol_all, sub_all, row_id_column_name)), 6
        )

    return out


In [ ]:
from fastai.vision.all import load_learner
from tqdm.auto import tqdm
import kagglehub

COMPETITION_DIR = Path(
    "/kaggle/input/competitions/umud-challenge-muscle-architecture-in-ultrasound-data"
)
if not COMPETITION_DIR.exists():
    COMPETITION_DIR = Path(
        kagglehub.competition_download("umud-challenge-muscle-architecture-in-ultrasound-data")
    )
COMP_DIRS = {
    "apo_img": COMPETITION_DIR / "apo_imgs_v1/apo_images_new_model_v1",
    "apo_mask": COMPETITION_DIR / "apo_masks_v1/apo_masks_new_model_v1",
    "fasc_img": COMPETITION_DIR / "fasc_imgs_v1/fasc_images_new_model_v1",
    "fasc_mask": COMPETITION_DIR / "fasc_masks_v1/fasc_masks_new_model_v1",
}
IMAGE_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}


def build_lookup(directory: Path) -> dict[str, Path]:
    if not directory.exists():
        return {}
    return {
        p.name: p
        for p in directory.rglob("*")
        if p.suffix.lower() in IMAGE_EXTS and p.name != "Thumbs.db"
    }


def resolve_pkl(preferred: list[Path], filename: str) -> Path:
    for p in preferred:
        if p.exists():
            return p
    hits = sorted(Path("/kaggle/input").rglob(filename))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")


def resolve_filename(stem: str, stem_to_filename: dict[str, str], lookups: dict[str, dict[str, Path]]) -> str | None:
    filename = stem_to_filename.get(stem)
    if filename and filename in lookups["apo_img"]:
        return filename
    for cand in (f"{stem}.tif", f"{stem}.png", f"{stem}.jpg"):
        if cand in lookups["apo_img"]:
            return cand
    return None


comp_lookups = {k: build_lookup(v) for k, v in COMP_DIRS.items()}
print({k: len(v) for k, v in comp_lookups.items()})
assert comp_lookups["apo_img"], f"Competition apo images not found under {COMPETITION_DIR}"

manifest_path = resolve_subdir(DATASET_ROOT, "manifests") / "train_apo_gray55_line.csv"
manifest = pd.read_csv(manifest_path)
stem_to_filename = dict(zip(manifest["stem"].astype(str), manifest["filename"].astype(str)))

fasc_model_path = resolve_pkl(
    [Path("/kaggle/input/notebooks/ucheozoemena/umud-train-mounted-phase-3/fasc_baseline.pkl")],
    "fasc_baseline.pkl",
)
fasc_learn = load_learner(fasc_model_path)
print("Fasc model:", fasc_model_path)

val_stems = [Path(f).stem for f in dls.valid.items]
print(f"Val images: {len(val_stems)}")

gt_rows, pred_rows = [], []
skip_manifest = skip_fasc = 0
for stem in tqdm(val_stems, desc="val umud"):
    filename = resolve_filename(stem, stem_to_filename, comp_lookups)
    if not filename:
        skip_manifest += 1
        continue
    if filename not in comp_lookups["fasc_mask"]:
        skip_fasc += 1
        continue
    img = load_gray(comp_lookups["apo_img"][filename])
    fasc_raw = load_mask(comp_lookups["fasc_mask"][filename])
    apo_raw = load_mask(comp_lookups["apo_mask"][filename])
    gt = gt_geometry_from_masks(fasc_raw, apo_raw, img.shape, MM_PER_PIXEL)
    gt_rows.append({"image_id": filename, **gt})
    pred = predict_geometry(img, fasc_learn, learn, IMG_SIZE, MM_PER_PIXEL)
    pred_rows.append(
        {
            "image_id": filename,
            "pa_deg": pred["pa_deg"],
            "fl_mm": pred["fl_mm"],
            "mt_mm": pred["mt_mm"],
            "mt_fail_reason": pred.get("mt_fail_reason"),
            "apo_cov": pred.get("apo_cov"),
        }
    )

print(f"Scored {len(pred_rows)}/{len(val_stems)} val images (skip_manifest={skip_manifest}, skip_fasc={skip_fasc})")

gt_df = pd.DataFrame(gt_rows)
pred_df = pd.DataFrame(pred_rows)
if len(pred_df) == 0:
    summary = {
        "n_total": 0,
        "n_pred_finite": 0,
        "n_gt_finite": 0,
        "n_scorable": 0,
        "val_mt_ok_pct": float("nan"),
        "val_umud_score": float("nan"),
        "val_umud_score_strict": float("nan"),
    }
    print("WARN: no dual-track val images scored — check competition mount + manifest stems")
else:
    pred_submit = pred_df[["image_id", "pa_deg", "fl_mm", "mt_mm"]]
    summary = score_summary(gt_df, pred_submit, row_id_column_name="image_id")
    print("Val UMUD summary:", summary)
    display(local_metric_report(gt_df, pred_submit))
    if pred_df["mt_mm"].isna().any():
        display(pred_df.loc[pred_df["mt_mm"].isna(), ["image_id", "mt_fail_reason", "apo_cov"]])
        display(pred_df.loc[pred_df["mt_mm"].isna(), "mt_fail_reason"].value_counts())

row = timing.iloc[0].to_dict()
for col in ("val_umud_score", "val_umud_score_strict", "val_mt_ok_pct", "n_scorable", "n_total"):
    row[col] = summary.get(col, float("nan"))
timing = pd.DataFrame([row])
timing.to_csv(WORKING / "val_umud_report.csv", index=False)
timing.to_csv(WORKING / "timing_report.csv", index=False)
display(timing)
